# Equity Derivatives RAG — Part 2: Agent & Evaluation

This notebook creates the Cortex Agent combining structured and unstructured tools,
builds the ground-truth evaluation dataset, and runs agent evaluations.

**Prerequisites:** Complete the `01_document_intelligence` notebook first.

| Step | What it does |
|------|--------------|
| 1 | Create views over Snowflake Public Data (Free) marketplace data |
| 2 | Create semantic views for stock prices, SEC financials, FX rates |
| 3 | Create the `EQ_DERIVATIVES_AGENT` with 5 tools |
| 4 | Test agent in Snowsight UI |
| 5 | Build 16-query ground-truth evaluation dataset |
| 6 | Register as a Snowflake Dataset |
| 7 | Run agent evaluation |

In [ ]:
%%sql -r setup_context
USE DATABASE CORTEX_AI_HOL;
USE SCHEMA RAG_PIPELINE;
USE WAREHOUSE COMPUTE_WH;

## Step 1: Marketplace Data Setup

Creates views over the **Snowflake Public Data (Free)** marketplace listing (`SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE`). Uses Cybersyn's EAV model: pivot `STOCK_PRICE_TIMESERIES`, `SEC_CORPORATE_REPORT_ATTRIBUTES`, and `FX_RATES_TIMESERIES` into flat, query-friendly views.

In [ ]:
%%sql -r step_1
-- Install "Snowflake Public Data (Free)" from Marketplace:
--   https://app.snowflake.com/marketplace/listing/GZTSZ290BV255
--
-- After installation the shared database is named: SNOWFLAKE_PUBLIC_DATA_FREE
-- Schema: PUBLIC_DATA_FREE
--
-- The data uses Cybersyn's EAV model:
--   STOCK_PRICE_TIMESERIES    — daily OHLC + volume per ticker / variable
--   SEC_CORPORATE_REPORT_ATTRIBUTES — XBRL financials (revenue, net income, etc.)
--   FX_RATES_TIMESERIES       — FX rates by base/quote currency pair
--   COMPANY_INDEX             — ticker ↔ company name / exchange mapping

-- ── Stock prices: pivot EAV → one row per ticker/date with close + volume ──
CREATE OR REPLACE VIEW V_STOCK_PRICES AS
SELECT
    p.TICKER,
    c.COMPANY_NAME,
    p.DATE::DATE                                        AS price_date,
    MAX(CASE WHEN p.VARIABLE = 'all-day_high'               THEN p.VALUE END) AS high,
    MAX(CASE WHEN p.VARIABLE = 'all-day_low'                THEN p.VALUE END) AS low,
    MAX(CASE WHEN p.VARIABLE = 'post-market_close'          THEN p.VALUE END) AS close,
    MAX(CASE WHEN p.VARIABLE = 'post-market_close_adjusted' THEN p.VALUE END) AS close_adjusted,
    MAX(CASE WHEN p.VARIABLE = 'nasdaq_volume'              THEN p.VALUE END) AS volume
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.STOCK_PRICE_TIMESERIES p
LEFT JOIN SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.COMPANY_INDEX c
  ON p.TICKER = c.PRIMARY_TICKER
WHERE p.DATE::DATE >= DATEADD('year', -2, CURRENT_DATE())
  -- Tickers covered by equity research reports in this lab
  -- Tickers auto-populated from equity research corpus (adapts to new reports)
  AND p.TICKER IN (
      SELECT DISTINCT UPPER(primary_ticker)
      FROM CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_EXTRACTED
      WHERE primary_ticker IS NOT NULL
        AND primary_ticker NOT IN ('N/A', '', 'SECTOR')
  )
GROUP BY p.TICKER, c.COMPANY_NAME, p.DATE::DATE;

-- ── SEC financials: pivot XBRL tags → revenue & net income per company/period ──
CREATE OR REPLACE VIEW V_SEC_FINANCIALS AS
SELECT
    ci.PRIMARY_TICKER                                   AS ticker,
    ci.COMPANY_NAME,
    f.FORM_TYPE,
    f.PERIOD_END_DATE::DATE                             AS period_end_date,
    MAX(CASE WHEN f.TAG = 'Revenues'
          OR  f.TAG = 'RevenueFromContractWithCustomerExcludingAssessedTax'
             THEN TRY_TO_NUMBER(f.VALUE) END)           AS revenue,
    MAX(CASE WHEN f.TAG = 'NetIncomeLoss'
             THEN TRY_TO_NUMBER(f.VALUE) END)           AS net_income,
    MAX(CASE WHEN f.TAG = 'Assets'
             THEN TRY_TO_NUMBER(f.VALUE) END)           AS total_assets,
    MAX(CASE WHEN f.TAG = 'Liabilities'
             THEN TRY_TO_NUMBER(f.VALUE) END)           AS total_liabilities
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.SEC_CORPORATE_REPORT_ATTRIBUTES f
JOIN SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.SEC_CIK_INDEX                   ck
  ON f.CIK = ck.CIK
JOIN SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.COMPANY_INDEX                   ci
  ON ck.COMPANY_ID = ci.COMPANY_ID
WHERE f.FORM_TYPE IN ('10-K', '10-Q')
  AND f.PERIOD_END_DATE::DATE >= DATEADD('year', -2, CURRENT_DATE())
  AND f.TAG IN (
      'Revenues',
      'RevenueFromContractWithCustomerExcludingAssessedTax',
      'NetIncomeLoss',
      'Assets',
      'Liabilities'
  )
  AND ci.PRIMARY_TICKER IN (
      SELECT DISTINCT UPPER(primary_ticker)
      FROM CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_EXTRACTED
      WHERE primary_ticker IS NOT NULL
        AND primary_ticker NOT IN ('N/A', '', 'SECTOR')
  )
GROUP BY ci.PRIMARY_TICKER, ci.COMPANY_NAME, f.FORM_TYPE, f.PERIOD_END_DATE::DATE;

-- ── FX rates: filter to major currency pairs relevant for ISDA agreements ──
CREATE OR REPLACE VIEW V_FX_RATES AS
SELECT
    DATE::DATE          AS rate_date,
    BASE_CURRENCY_ID    AS base_currency,
    QUOTE_CURRENCY_ID   AS quote_currency,
    VALUE               AS rate
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FX_RATES_TIMESERIES
WHERE DATE::DATE >= DATEADD('year', -1, CURRENT_DATE())
  AND QUOTE_CURRENCY_ID IN ('USD', 'GBP', 'EUR', 'JPY')
  AND BASE_CURRENCY_ID  IN ('USD', 'GBP', 'EUR', 'JPY')
  AND BASE_CURRENCY_ID != QUOTE_CURRENCY_ID;

-- Verify views return data
SELECT 'V_STOCK_PRICES'   AS view_name, COUNT(*) AS row_count FROM V_STOCK_PRICES   UNION ALL
SELECT 'V_SEC_FINANCIALS'  AS view_name, COUNT(*) AS row_count FROM V_SEC_FINANCIALS  UNION ALL
SELECT 'V_FX_RATES'       AS view_name, COUNT(*) AS row_count FROM V_FX_RATES;

## Step 2: Semantic Views

Registers semantic views for Cortex Analyst over the three marketplace views plus a FX rates view. The ISDA semantic view is created separately via the Snowsight UI (see README).

In [ ]:
%%sql -r step_2
-- ============================================================================
-- STEP 2: CREATE SEMANTIC VIEWS
-- ============================================================================
-- Note: Run this AFTER completing notebook 01_document_intelligence
-- which creates ISDA_AGREEMENT_TERMS, V_STOCK_PRICES, V_SEC_FINANCIALS, V_FX_RATES

-- Semantic View: ISDA Agreement Terms (created here — NOT needed in Snowsight UI)
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'CORTEX_AI_HOL.RAG_PIPELINE',
  $$
name: isda_agreement_terms_sv
description: Structured ISDA Master Agreement and Amendment data. Contains parties, governing law, cross-default thresholds, close-out methods, events of default, and termination events. Use to answer questions about specific clause values, counterparty terms, or agreement comparisons. Parties include Barclays Bank PLC, Bank of America N.A., Royal Bank of Scotland, and Comerica Bank.

tables:
  - name: agreement_terms
    description: Extracted ISDA agreement terms — 4 master agreements plus 1 amendment
    base_table:
      database: CORTEX_AI_HOL
      schema: RAG_PIPELINE
      table: ISDA_AGREEMENT_TERMS

    dimensions:
      - name: filename
        description: Source document filename
        expr: FILENAME
        data_type: VARCHAR
      - name: document_type
        description: MASTER_AGREEMENT or AMENDMENT
        expr: DOCUMENT_TYPE
        data_type: VARCHAR
        sample_values: ["MASTER_AGREEMENT", "AMENDMENT"]
      - name: party_a
        description: Party A — the dealer/bank side. Known values include Barclays Bank PLC, Bank of America N.A., Royal Bank of Scotland PLC, Comerica Bank.
        expr: PARTY_A
        data_type: VARCHAR
        synonyms: [counterparty a, dealer, bank, sell side]
      - name: party_b
        description: Party B — the client/buy-side. Known values include Deutsche Bank Trust Company Delaware, LKQ Corporation, Rackspace US Inc, World Omni Auto Receivables Trust.
        expr: PARTY_B
        data_type: VARCHAR
        synonyms: [counterparty b, client, buy side, fund]
      - name: agreement_version
        description: ISDA version year — 1992 or 2002
        expr: AGREEMENT_VERSION
        data_type: VARCHAR
        sample_values: ["1992", "2002"]
      - name: governing_law
        description: Legal jurisdiction governing the agreement
        expr: GOVERNING_LAW
        data_type: VARCHAR
        sample_values: ["English law", "New York law"]
      - name: cross_default_applicable
        description: Whether cross-default provisions apply
        expr: CROSS_DEFAULT_APPLICABLE
        data_type: VARCHAR
      - name: cross_default_currency
        description: Currency of the cross-default threshold (USD, GBP, EUR)
        expr: CROSS_DEFAULT_CURRENCY
        data_type: VARCHAR
      - name: aet_party_a
        description: Whether Automatic Early Termination applies to Party A
        expr: AET_PARTY_A
        data_type: VARCHAR
        synonyms: [automatic early termination party a]
      - name: aet_party_b
        description: Whether Automatic Early Termination applies to Party B
        expr: AET_PARTY_B
        data_type: VARCHAR
      - name: closeout_method
        description: Close-out calculation method — Market Quotation, Loss, or Close-out Amount
        expr: CLOSEOUT_METHOD
        data_type: VARCHAR
        synonyms: [close out, termination payment method]
      - name: events_of_default
        description: Events of Default from Section 5(a) as comma-separated list
        expr: EVENTS_OF_DEFAULT
        data_type: VARCHAR
        synonyms: [default events, section 5a]
      - name: termination_events
        description: Termination Events from Section 5(b) as comma-separated list
        expr: TERMINATION_EVENTS
        data_type: VARCHAR
        synonyms: [section 5b]

    time_dimensions:
      - name: effective_date
        description: Date the agreement or amendment was executed
        expr: TRY_TO_DATE(EFFECTIVE_DATE)
        data_type: DATE

    facts:
      - name: cross_default_threshold
        description: Cross-default threshold amount. Common values range from 100000 to 50000000.
        expr: CROSS_DEFAULT_THRESHOLD
        data_type: NUMBER
        synonyms: [threshold amount, default threshold, cross default amount]

    metrics:
      - name: agreement_count
        description: Total number of ISDA agreements
        expr: COUNT(FILENAME)
  $$,
  FALSE
);

-- Semantic View: Stock Prices
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'CORTEX_AI_HOL.RAG_PIPELINE',
  $$
name: stock_market_data
description: Daily stock prices for companies covered by equity research. Use to compare analyst price targets against actual performance and analyze price trends.

tables:
  - name: stock_prices
    description: Daily OHLC prices and volume, one row per ticker per day
    base_table:
      database: CORTEX_AI_HOL
      schema: RAG_PIPELINE
      table: V_STOCK_PRICES

    dimensions:
      - name: ticker
        description: Stock ticker symbol (e.g. AAPL, NVDA, JPM)
        expr: TICKER
        data_type: VARCHAR
        synonyms: [symbol, stock]
      - name: company_name
        description: Full company name
        expr: COMPANY_NAME
        data_type: VARCHAR

    time_dimensions:
      - name: price_date
        description: Trading date
        expr: PRICE_DATE
        data_type: DATE

    facts:
      - name: close
        description: Post-market closing price in USD
        expr: CLOSE
        data_type: NUMBER
        synonyms: [price, closing price, stock price]
      - name: close_adjusted
        description: Split/dividend-adjusted closing price
        expr: CLOSE_ADJUSTED
        data_type: NUMBER
      - name: high
        description: All-day high price
        expr: HIGH
        data_type: NUMBER
      - name: low
        description: All-day low price
        expr: LOW
        data_type: NUMBER
      - name: volume
        description: Daily trading volume (shares)
        expr: VOLUME
        data_type: NUMBER

    metrics:
      - name: latest_close
        description: Most recent closing price
        expr: MAX(CLOSE)
      - name: avg_volume
        description: Average daily trading volume
        expr: AVG(VOLUME)
      - name: price_return_pct
        description: Percentage return — (max close - min close) / min close * 100
        expr: (MAX(CLOSE) - MIN(CLOSE)) / NULLIF(MIN(CLOSE), 0) * 100
  $$,
  FALSE
);

-- Semantic View: SEC Financials
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'CORTEX_AI_HOL.RAG_PIPELINE',
  $$
name: sec_financials_data
description: Quarterly and annual SEC filing financials (10-K / 10-Q) for companies covered by equity research. Contains revenue, net income, assets, and liabilities parsed from XBRL.

tables:
  - name: sec_financials
    description: Key financial metrics from SEC XBRL filings, one row per company per reporting period
    base_table:
      database: CORTEX_AI_HOL
      schema: RAG_PIPELINE
      table: V_SEC_FINANCIALS

    dimensions:
      - name: ticker
        description: Stock ticker symbol
        expr: TICKER
        data_type: VARCHAR
      - name: company_name
        description: Full company name
        expr: COMPANY_NAME
        data_type: VARCHAR
      - name: form_type
        description: SEC filing type — 10-K (annual) or 10-Q (quarterly)
        expr: FORM_TYPE
        data_type: VARCHAR
        sample_values: ["10-K", "10-Q"]

    time_dimensions:
      - name: period_end_date
        description: End date of the fiscal period covered by the filing
        expr: PERIOD_END_DATE
        data_type: DATE

    facts:
      - name: revenue
        description: Total revenue reported in the filing (USD)
        expr: REVENUE
        data_type: NUMBER
        synonyms: [sales, top line]
      - name: net_income
        description: Net income (profit or loss) reported (USD)
        expr: NET_INCOME
        data_type: NUMBER
        synonyms: [profit, earnings, bottom line, net profit]
      - name: total_assets
        description: Total assets on the balance sheet (USD)
        expr: TOTAL_ASSETS
        data_type: NUMBER
      - name: total_liabilities
        description: Total liabilities on the balance sheet (USD)
        expr: TOTAL_LIABILITIES
        data_type: NUMBER

    metrics:
      - name: profit_margin_pct
        description: Net income as a percentage of revenue
        expr: AVG(NET_INCOME / NULLIF(REVENUE, 0) * 100)
      - name: latest_revenue
        description: Most recent revenue figure
        expr: MAX(REVENUE)
  $$,
  FALSE
);

-- Semantic View: FX Rates
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'CORTEX_AI_HOL.RAG_PIPELINE',
  $$
name: fx_rates_data
description: Daily foreign exchange rates between major currency pairs. Relevant for ISDA agreements that reference cross-default thresholds in different currencies (USD, GBP, EUR, JPY).

tables:
  - name: fx_rates
    description: Daily FX spot rates between major currencies
    base_table:
      database: CORTEX_AI_HOL
      schema: RAG_PIPELINE
      table: V_FX_RATES

    dimensions:
      - name: base_currency
        description: The base currency code (e.g. USD, GBP)
        expr: BASE_CURRENCY
        data_type: VARCHAR
        synonyms: [from currency]
      - name: quote_currency
        description: The quote/target currency code (e.g. EUR, JPY)
        expr: QUOTE_CURRENCY
        data_type: VARCHAR
        synonyms: [to currency]

    time_dimensions:
      - name: rate_date
        description: Date of the exchange rate
        expr: RATE_DATE
        data_type: DATE

    facts:
      - name: rate
        description: Exchange rate — units of quote currency per 1 unit of base currency
        expr: RATE
        data_type: NUMBER
        synonyms: [exchange rate, spot rate, fx rate]

    metrics:
      - name: latest_rate
        description: Most recent exchange rate
        expr: MAX(RATE)
  $$,
  FALSE
);

## Step 3: Create the Agent

Deploys `EQ_DERIVATIVES_AGENT` with **5 tools**: ISDA structured data, ISDA full-text search, equity research multimodal search, stock prices, and SEC financials. Agent persona: equity derivatives salesperson on a client call.

## Step 3a: Equity Research Search — Stored Procedure Custom Tool

The `EQUITY_RESEARCH_SEARCH_AGENT` CSS uses managed arctic embeddings over both
`searchable_text` and `chart_content` (chart visual descriptions), so the agent
can retrieve relevant research pages with a plain `query` — no pre-computed vector needed.

We wrap it in a stored procedure to expose optional **ticker filtering**, **numeric boost**
(popular reports rank higher), and **time decay** (recent reports rank higher) as a single
`generic` type agent tool: `EQUITY_RESEARCH_MULTIMODAL_SEARCH(query, ticker, limit)`.

In [ ]:
%%sql -r create_multimodal_proc
-- ============================================================================
-- CUSTOM TOOL: Equity Research Search Stored Procedure
-- ============================================================================
-- Wraps EQUITY_RESEARCH_SEARCH_AGENT (arctic managed embeddings on
-- searchable_chunk) with optional primary_ticker filtering, numeric boost,
-- and time decay — exposed as a single generic agent tool.
CREATE OR REPLACE PROCEDURE CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_MULTIMODAL_SEARCH(
    QUERY   VARCHAR,
    TICKER  VARCHAR DEFAULT NULL,
    LIMIT_N INTEGER DEFAULT 5
)
RETURNS VARIANT
LANGUAGE SQL
AS
$$
DECLARE
    search_json VARCHAR;
    results     VARIANT;
BEGIN
    IF (:TICKER IS NOT NULL AND :TICKER != '') THEN
        SELECT TO_JSON(OBJECT_CONSTRUCT(
            'query',   :QUERY,
            'filter',  OBJECT_CONSTRUCT('@eq', OBJECT_CONSTRUCT('primary_ticker', UPPER(:TICKER))),
            'columns', ARRAY_CONSTRUCT('filename','primary_ticker','report_type','rating',
                                       'analyst_name','publish_date','view_count',
                                       'chunk_index','chunk_text','searchable_chunk'),
            'limit',   :LIMIT_N,
            'boosts',  OBJECT_CONSTRUCT('view_count', OBJECT_CONSTRUCT('boost_by','multiplier')),
            'decays',  OBJECT_CONSTRUCT('publish_date', OBJECT_CONSTRUCT('decay_speed','moderate'))
        )) INTO :search_json;
    ELSE
        SELECT TO_JSON(OBJECT_CONSTRUCT(
            'query',   :QUERY,
            'columns', ARRAY_CONSTRUCT('filename','primary_ticker','report_type','rating',
                                       'analyst_name','publish_date','view_count',
                                       'chunk_index','chunk_text','searchable_chunk'),
            'limit',   :LIMIT_N,
            'boosts',  OBJECT_CONSTRUCT('view_count', OBJECT_CONSTRUCT('boost_by','multiplier')),
            'decays',  OBJECT_CONSTRUCT('publish_date', OBJECT_CONSTRUCT('decay_speed','moderate'))
        )) INTO :search_json;
    END IF;

    SELECT PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_SEARCH_AGENT',
        :search_json
    )) INTO :results;

    RETURN :results;
END;
$$;

GRANT USAGE ON PROCEDURE CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_MULTIMODAL_SEARCH(VARCHAR, VARCHAR, INTEGER)
  TO ROLE PUBLIC;

-- Test the procedure
CALL EQUITY_RESEARCH_MULTIMODAL_SEARCH('autonomous vehicles EV market growth charts', NULL, 3);


In [ ]:
%%sql -r step_3
-- ============================================================================
-- STEP 3: CREATE THE CORTEX AGENT
-- ============================================================================
-- 6 tools: 4 Cortex Analyst (structured) + 1 generic stored proc + 1 Cortex Search
-- All execution_environment blocks are required for Cortex Analyst tools
CREATE OR REPLACE AGENT EQ_DERIVATIVES_AGENT
  COMMENT = 'Equity derivatives sales agent — ISDA agreements, equity research, market data'
  FROM SPECIFICATION
$$
models:
  orchestration: claude-sonnet-4-5

orchestration:
  budget:
    seconds: 180
    tokens: 100000

instructions:
  system: |
    You are an AI Assistant for an equity derivatives sales desk. You help salespeople
    quickly answer client questions on calls — be concise, precise, and cite sources.

    ## What you have access to

    **ISDA Agreements (structured)**
    - Tool: AgreementTerms
    - 5 agreements: 4 master agreements + 1 amendment
    - Parties: Barclays Bank PLC, Bank of America N.A., Royal Bank of Scotland, Comerica Bank
    - Fields: governing law, cross-default threshold/currency, close-out method, AET, events of default
    - Use for: threshold amounts, party AET status, governing law, netting provisions

    **ISDA Document Text (unstructured)**
    - Tool: DocumentSearch_ISDA
    - Full parsed text of all ISDA PDFs, page-level indexing
    - Attribute filters: party_a, party_b, document_type, governing_law, page_number
    - Use for: exact wording, definitions, clause language not in structured data

    **Equity Research + Macro Reports (multimodal)**
    - Tool: EquityResearch (stored procedure using voyage-multimodal-3)
    - 19 reports: 10 synthetic equity + 9 real GS research (macro, thematic, sector)
    - Date range: 2024-2026
    - Supports: ticker filter, readership boost, time decay
    - Use for: analyst ratings, price targets, thematic research, chart/visual content

    **Stock Market Data (structured)**
    - Tool: MarketData
    - Daily prices for Equity Stocks (last 2 years)
    - Use for: current/historical prices, computing returns, price vs target comparison

    **SEC Filings — Financials (structured)**
    - Tool: SECFilings
    - 10-K/10-Q financials for (last 2 years)
    - Fields: revenue, net_income, total_assets, total_liabilities
    - Use for: revenue, earnings, profit margin, balance sheet

    **FX Rates (structured)**
    - Tool: FXRates
    - Daily USD/GBP/EUR/JPY spot rates (last 12 months)
    - Use for: converting ISDA cross-default thresholds between currencies

    ## Routing rules
    - ISDA clause values → AgreementTerms first
    - ISDA wording/definitions → DocumentSearch_ISDA
    - Equity or macro research → EquityResearch (accepts optional ticker filter)
    - Stock prices/returns → MarketData
    - Revenue/earnings → SECFilings
    - FX conversions → FXRates
    - Always cite source; format currency with symbols ($, £, €)

    ## ISDA domain knowledge
    - **Close-out calculation method**: 2002 ISDA agreements use **Close-out Amount**.
      1992 agreements use **Market Quotation** or **Loss**. If structured data shows
      a different value for a 2002 agreement, flag it as a potential data quality issue
      and state the correct default for a 2002 agreement.
    - **Netting**: Section 2(c) governs payment netting. Always verify which version
      (1992 or 2002) applies before citing close-out provisions.

  orchestration: |
    Multi-tool patterns:
    - "Is NVDA above its price target?" → EquityResearch (get target) + MarketData (current price) → delta
    - "Convert Barclays threshold to USD" → AgreementTerms (threshold + currency) + FXRates (rate)
    - "What does GS say about autonomous vehicles?" → EquityResearch (topic search, no ticker filter)

  response: |
    Answers for salespeople on client calls:
    - Front-load the answer, then add context
    - Include exact values, currencies, dates
    - Always state which document or data source the answer comes from
    - For price vs target: state delta and direction ("$17.82 (10.2%) above target")
    - For ISDA: always name the counterparties the answer applies to
    - For charts/visualizations: describe the chart type (bar, line, scatter, etc.),
      what data is on each axis, the key trend shown, and cite the source report
    - When returning results from DocumentSearch_ISDA, always include the source
      `filename` (e.g. 'Source: ISDA_Master_Agreement_Barclays.pdf, page 3')

tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: AgreementTerms
      description: "Query structured ISDA agreement data. Contains 5 agreements with
        fields: party_a (Barclays, Bank of America, RBS, Comerica), party_b (client names),
        governing_law, cross_default_threshold, cross_default_currency, closeout_method,
        aet_party_a, aet_party_b, events_of_default, termination_events, netting_applicable.
        Use for specific clause values and agreement comparisons. Do NOT use for verbatim
        clause text — use DocumentSearch_ISDA for that."

  - tool_spec:
      type: cortex_search
      name: DocumentSearch_ISDA
      description: "Full-text page-level search over raw ISDA PDFs. Returns verbatim passages.
        Attribute filters: party_a, party_b, document_type (MASTER_AGREEMENT or AMENDMENT),
        governing_law (English law or New York law), page_number. Use for definitions,
        exact wording, and provisions not in structured fields."

  - tool_spec:
      type: generic
      name: EquityResearch
      description: "Multimodal search over 19 research reports (equity + macro/thematic).
        Embeds query with voyage-multimodal-3 for cross-modal retrieval — text queries find
        visually similar chart content. Accepts: query (required), ticker (optional filter,
        e.g. NVDA), limit_n (default 5). Results boosted by readership and time-decayed.
        Use for ratings, price targets, investment thesis, thematic research, chart analysis."
      input_schema:
        type: object
        properties:
          query:
            type: string
            description: "Natural language search query"
          ticker:
            type: string
            description: "Optional: filter by specific ticker (e.g. NVDA, AAPL)"
          limit_n:
            type: integer
            description: "Number of results to return (default 5)"
        required: [query]

  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: MarketData
      description: "Daily stock prices for US equities covered by reports in the equity research corpus.
        Fields: ticker, company_name, price_date, high, low, close, close_adjusted, volume.
        Last 2 years of data (coverage may not extend to today). When returning the most recent
        price, always include the price_date and note if data appears stale. Use for
        current/historical prices and computing returns."

  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: SECFilings
      description: "Quarterly and annual 10-K/10-Q financials for US equities covered
        by reports in the equity research corpus. Fields: ticker, form_type, period_end_date, revenue,
        net_income, total_assets, total_liabilities. Last 2 years of XBRL data."

  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: FXRates
      description: "Daily FX spot rates between USD, GBP, EUR, JPY for the last 12 months.
        Fields: rate_date, base_currency, quote_currency, rate. Use for converting ISDA
        cross-default thresholds between currencies."

tool_resources:
  AgreementTerms:
    semantic_view: "CORTEX_AI_HOL.RAG_PIPELINE.ISDA_AGREEMENT_TERMS_SV"
    execution_environment:
      type: warehouse
      warehouse: "COMPUTE_WH"
  DocumentSearch_ISDA:
    name: "CORTEX_AI_HOL.RAG_PIPELINE.ISDA_DOCUMENT_SEARCH"
    max_results: 5
  EquityResearch:
    identifier: "CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_MULTIMODAL_SEARCH"
    type: procedure
    execution_environment:
      type: warehouse
      warehouse: "COMPUTE_WH"
  MarketData:
    semantic_view: "CORTEX_AI_HOL.RAG_PIPELINE.STOCK_MARKET_DATA"
    execution_environment:
      type: warehouse
      warehouse: "COMPUTE_WH"
  SECFilings:
    semantic_view: "CORTEX_AI_HOL.RAG_PIPELINE.SEC_FINANCIALS_DATA"
    execution_environment:
      type: warehouse
      warehouse: "COMPUTE_WH"
  FXRates:
    semantic_view: "CORTEX_AI_HOL.RAG_PIPELINE.FX_RATES_DATA"
    execution_environment:
      type: warehouse
      warehouse: "COMPUTE_WH"
$$;

SHOW AGENTS IN CORTEX_AI_HOL.RAG_PIPELINE;

## Step 3b: Expose the Agent as an MCP Server

`CREATE MCP SERVER` wraps the agent as a **Model Context Protocol endpoint** — the
open standard that lets external tools (Claude Desktop, Cursor, LangGraph, etc.)
discover and invoke the agent's capabilities without any custom integration code.

The MCP server is accessible at:
```
https://<ORG>-<ACCOUNT>.snowflakecomputing.com/api/v2/databases/CORTEX_AI_HOL/schemas/RAG_PIPELINE/mcp-servers/EQ_DERIVATIVES_MCP_SERVER
```

**Connecting from Claude Desktop or Cursor** — add this block to your `mcp.json`
(use a Programmatic Access Token for auth; OAuth DCR is not yet supported):
```json
{
  "mcpServers": {
    "eq-derivatives": {
      "url": "https://<ORG>-<ACCOUNT>.snowflakecomputing.com/api/v2/databases/CORTEX_AI_HOL/schemas/RAG_PIPELINE/mcp-servers/EQ_DERIVATIVES_MCP_SERVER",
      "headers": {
        "Authorization": "Bearer <YOUR_PAT>"
      }
    }
  }
}
```
> Use **hyphens** in the hostname (`ORG-ACCOUNT`), not underscores.
> Generate a PAT in Snowsight: **Admin → Security → Programmatic Access Tokens**.

In [ ]:
%%sql -r create_mcp_server
-- ============================================================================
-- EXPOSE EQ_DERIVATIVES_AGENT AS AN MCP SERVER
-- ============================================================================
-- CREATE MCP SERVER wraps Snowflake-native objects as a Model Context Protocol
-- endpoint.  External clients (Claude Desktop, Cursor, LangGraph, etc.) connect
-- to this URL and discover the agent as a callable tool — no custom integration.
CREATE OR REPLACE MCP SERVER CORTEX_AI_HOL.RAG_PIPELINE.EQ_DERIVATIVES_MCP_SERVER
  FROM SPECIFICATION $$
    tools:
      - name: "eq-derivatives-agent"
        type: "CORTEX_AGENT_RUN"
        identifier: "CORTEX_AI_HOL.RAG_PIPELINE.EQ_DERIVATIVES_AGENT"
        title: "Equity Derivatives Assistant"
        description: >
          AI assistant for equity derivatives sales. Answers questions about
          ISDA master agreements, equity research reports, market data, and
          SEC filings. Use for counterparty analysis, covenant lookup,
          analyst ratings, price targets, and cross-asset research.
  $$;

GRANT USAGE ON MCP SERVER CORTEX_AI_HOL.RAG_PIPELINE.EQ_DERIVATIVES_MCP_SERVER
  TO ROLE PUBLIC;

-- Show the MCP server and its endpoint details
DESCRIBE MCP SERVER CORTEX_AI_HOL.RAG_PIPELINE.EQ_DERIVATIVES_MCP_SERVER;

## Step 4: Test the Agent

**Run in Snowsight UI**: AI & ML → Agents → EQ_DERIVATIVES_AGENT → Chat. This cell contains the sample prompts — no SQL to execute here.

## Step 4a: RBAC — Grant Access to All Objects

Grant `USAGE` on the agent and all its dependencies to `PUBLIC` so any role in the account can use the agent in Snowsight.

On a trial account: run this as `ACCOUNTADMIN` or `SYSADMIN`.

In [ ]:
%%sql -r rbac_grants
-- ============================================================================
-- RBAC: Grant access to all HOL objects for lab participants
-- ============================================================================
-- Run as ACCOUNTADMIN or SYSADMIN

-- Agent
GRANT USAGE ON AGENT CORTEX_AI_HOL.RAG_PIPELINE.EQ_DERIVATIVES_AGENT  TO ROLE PUBLIC;

-- Cortex Search Services
GRANT USAGE ON CORTEX SEARCH SERVICE CORTEX_AI_HOL.RAG_PIPELINE.ISDA_DOCUMENT_SEARCH        TO ROLE PUBLIC;
GRANT USAGE ON CORTEX SEARCH SERVICE CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_SEARCH       TO ROLE PUBLIC;
GRANT USAGE ON CORTEX SEARCH SERVICE CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_SEARCH_AGENT TO ROLE PUBLIC;

-- Semantic Views
GRANT SELECT ON SEMANTIC VIEW CORTEX_AI_HOL.RAG_PIPELINE.ISDA_AGREEMENT_TERMS_SV TO ROLE PUBLIC;
GRANT SELECT ON SEMANTIC VIEW CORTEX_AI_HOL.RAG_PIPELINE.STOCK_MARKET_DATA        TO ROLE PUBLIC;
GRANT SELECT ON SEMANTIC VIEW CORTEX_AI_HOL.RAG_PIPELINE.SEC_FINANCIALS_DATA       TO ROLE PUBLIC;
GRANT SELECT ON SEMANTIC VIEW CORTEX_AI_HOL.RAG_PIPELINE.FX_RATES_DATA            TO ROLE PUBLIC;

-- Underlying views and tables
GRANT SELECT ON VIEW  CORTEX_AI_HOL.RAG_PIPELINE.V_STOCK_PRICES    TO ROLE PUBLIC;
GRANT SELECT ON VIEW  CORTEX_AI_HOL.RAG_PIPELINE.V_SEC_FINANCIALS   TO ROLE PUBLIC;
GRANT SELECT ON VIEW  CORTEX_AI_HOL.RAG_PIPELINE.V_FX_RATES         TO ROLE PUBLIC;
GRANT SELECT ON TABLE CORTEX_AI_HOL.RAG_PIPELINE.ISDA_AGREEMENT_TERMS TO ROLE PUBLIC;

-- Stored procedure (EquityResearch custom tool)
GRANT USAGE ON PROCEDURE CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_MULTIMODAL_SEARCH(VARCHAR, VARCHAR, INTEGER)
  TO ROLE PUBLIC;

-- Database, schema, warehouse
GRANT USAGE ON DATABASE  CORTEX_AI_HOL                  TO ROLE PUBLIC;
GRANT USAGE ON SCHEMA    CORTEX_AI_HOL.RAG_PIPELINE     TO ROLE PUBLIC;
GRANT USAGE ON WAREHOUSE COMPUTE_WH                     TO ROLE PUBLIC;
-- Cortex AI access (required for all AI functions)
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE PUBLIC;

SELECT 'RBAC grants applied successfully' AS status;

In [ ]:
%%sql -r step_4
/*
Test the agent in Snowsight:
  1. Navigate to AI & ML → Agents
  2. Select EQ_DERIVATIVES_AGENT
  3. Click "Chat" to open the agent playground
  4. Try these queries:

-- ISDA Structured:
"What is the cross-default threshold for each counterparty in our ISDA agreements?"

-- ISDA Unstructured:
"What does the ISDA agreement say about the definition of Early Termination Date?"

-- Equity Research:
"What is the analyst rating and price target for NVIDIA?"

-- Cross-tool:
"Which of our ISDA counterparties have research coverage, and what are their ratings?"
*/

## Step 5: Ground Truth Evaluation Dataset

Creates 16 ground-truth Q&A pairs covering all 5 tools. Each row has an `input_query` (VARCHAR) and a `ground_truth` (VARIANT) describing expected agent behavior.

In [ ]:
%%sql -r step_5
-- Create evaluation dataset for agent quality measurement

CREATE OR REPLACE TABLE AGENT_EVALUATION_DATA (
    input_query VARCHAR,
    ground_truth VARIANT
);

-- Insert ground truth questions spanning all tools
INSERT ALL
  -- ISDA Structured queries
  INTO AGENT_EVALUATION_DATA VALUES (
    'What is the cross-default threshold amount for Party A in our ISDA agreements?',
    PARSE_JSON('{"ground_truth_output": "The agent should query the AgreementTerms tool and return specific cross-default threshold amounts with currency for each agreement, citing whether the value comes from the original agreement or an amendment."}')
  )
  INTO AGENT_EVALUATION_DATA VALUES (
    'Which ISDA agreements are governed by English law?',
    PARSE_JSON('{"ground_truth_output": "The agent should query AgreementTerms filtering on governing_law and return agreements governed by English law with their party names and effective dates."}')
  )
  INTO AGENT_EVALUATION_DATA VALUES (
    'Is Automatic Early Termination applicable to any of our counterparties?',
    PARSE_JSON('{"ground_truth_output": "The agent should query AgreementTerms and return which parties have Automatic Early Termination enabled (true/false) for both Party A and Party B positions."}')
  )
  INTO AGENT_EVALUATION_DATA VALUES (
    'What close-out calculation method is used in our 2002 ISDA agreements?',
    PARSE_JSON('{"ground_truth_output": "The agent should identify 2002 ISDA agreements and report they use Close-out Amount as the calculation method, distinguishing from 1992 agreements which use Market Quotation or Loss."}')
  )
  -- ISDA Unstructured queries
  INTO AGENT_EVALUATION_DATA VALUES (
    'What is the exact definition of Events of Default in our ISDA master agreement?',
    PARSE_JSON('{"ground_truth_output": "The agent should use DocumentSearch_ISDA to find Section 5(a) and return the verbatim or near-verbatim definition of Events of Default. The substance of the answer (accurate clause text) is the primary criterion. Citing the source filename is expected but not the sole basis for scoring."}')
  )
  INTO AGENT_EVALUATION_DATA VALUES (
    'Search for provisions related to netting in our ISDA documentation',
    PARSE_JSON('{"ground_truth_output": "The agent should use DocumentSearch_ISDA to search for netting-related provisions and return relevant text passages. Ideally cites the source filename, but the primary criterion is accurate substantive content about payment netting, settlement netting, or Section 2(c) provisions."}')
  )
  -- Equity Research queries
  INTO AGENT_EVALUATION_DATA VALUES (
    'What is the current analyst rating and price target for NVIDIA?',
    PARSE_JSON('{"ground_truth_output": "The agent should use EquityResearch to find the most recent NVDA research report and return the rating (e.g., Buy/Overweight) and specific price target with currency."}')
  )
  INTO AGENT_EVALUATION_DATA VALUES (
    'Show me research reports about competitive threats in the semiconductor industry',
    PARSE_JSON('{"ground_truth_output": "The agent should use EquityResearch to search for semiconductor/competitive content and return relevant analyst commentary with source citations."}')
  )
  INTO AGENT_EVALUATION_DATA VALUES (
    'What charts or visualizations are available about revenue growth trends?',
    PARSE_JSON('{"ground_truth_output": "The agent should use EquityResearch to find reports containing revenue growth data or charts. The agent should describe the chart type and data trend if available from the indexed content, or summarize the revenue growth findings from the report. Citing the source report filename is expected."}')
  )
  -- Market Data queries
  INTO AGENT_EVALUATION_DATA VALUES (
    'What was Apple stock closing price on the most recent trading day?',
    PARSE_JSON('{"ground_truth_output": "The agent should use MarketData to query the latest available AAPL closing price and return the price with its date. If the data does not extend to today, the agent should explicitly state the data coverage period and note the limitation."}')
  )
  INTO AGENT_EVALUATION_DATA VALUES (
    'Compare the 3-month stock performance of NVDA vs AAPL',
    PARSE_JSON('{"ground_truth_output": "The agent should use MarketData to compute returns for both NVDA and AAPL over the last 3 months and present a comparison with percentage returns."}')
  )
  -- SEC Filings queries
  INTO AGENT_EVALUATION_DATA VALUES (
    'What was NVIDIA revenue in their most recent 10-K filing?',
    PARSE_JSON('{"ground_truth_output": "The agent should use SECFilings to find the most recent 10-K filing for NVDA and return the revenue figure with the filing date and period."}')
  )
  INTO AGENT_EVALUATION_DATA VALUES (
    'Which companies have the highest profit margin based on their latest filings?',
    PARSE_JSON('{"ground_truth_output": "The agent should use SECFilings to compute profit margins (net_income/revenue) across companies and rank them, returning the top companies with their margin percentages."}')
  )
  -- Cross-domain queries
  INTO AGENT_EVALUATION_DATA VALUES (
    'For NVIDIA, compare the analyst price target from equity research with the actual current stock price. Is the stock trading above or below target?',
    PARSE_JSON('{"ground_truth_output": "The agent should use both EquityResearch (for price target) and MarketData (for current price) and compute the delta, stating whether the stock is above or below the analyst target and by how much."}')
  )
  INTO AGENT_EVALUATION_DATA VALUES (
    'What counterparties in our ISDA agreements are publicly traded, and how have their stocks performed recently?',
    PARSE_JSON('{"ground_truth_output": "The agent should use AgreementTerms to identify counterparties (Barclays, Bank of America, RBS, Comerica), then attempt to look up their stock performance in MarketData. Full credit if agent correctly identifies counterparties AND explains that market data may not cover all bank holding companies. Partial credit for identifying counterparties even if market data lookup fails."}')
  )
  INTO AGENT_EVALUATION_DATA VALUES (
    'Summarize all available information about NVIDIA across our documents and market data',
    PARSE_JSON('{"ground_truth_output": "The agent should use multiple tools: EquityResearch for analyst views, MarketData for stock performance, SECFilings for fundamentals, and synthesize a comprehensive summary citing all sources."}')
  )
SELECT 1 FROM DUAL;

-- Verify
SELECT COUNT(*) AS total_queries, 
       COUNT(DISTINCT input_query) AS unique_queries
FROM AGENT_EVALUATION_DATA;

## Step 6: Register Evaluation Dataset

Calls `SYSTEM$CREATE_EVALUATION_DATASET` to register the table as a Snowflake Dataset object for the evaluation framework.

In [ ]:
%%sql -r step_6
-- ============================================================================
-- Register evaluation dataset
-- ============================================================================
-- SYSTEM$CREATE_EVALUATION_DATASET column_mapping keys:
--   'query_text'     → name of the input query column in the source table
--   'expected_tools' → name of the ground truth VARIANT column
-- Note: 'expected_tools' is the system function key — the column itself can
-- be named anything (here: GROUND_TRUTH).
CALL SYSTEM$CREATE_EVALUATION_DATASET(
  'Cortex Agent',
  'CORTEX_AI_HOL.RAG_PIPELINE.AGENT_EVALUATION_DATA',
  'CORTEX_AI_HOL.RAG_PIPELINE.EQ_DERIVATIVES_EVAL_DATASET',
  OBJECT_CONSTRUCT(
    'query_text',     'INPUT_QUERY',
    'expected_tools', 'GROUND_TRUTH'
  )
);

-- Verify: dataset should appear in agent Evaluations tab in Snowsight
-- (Navigate to AI & ML → Agents → EQ_DERIVATIVES_AGENT → Evaluations)
SHOW DATASETS IN SCHEMA CORTEX_AI_HOL.RAG_PIPELINE;

## Step 7: Run Agent Evaluation

Uploads `evaluation_config.yaml` to the `EVALUATION_CONFIG` stage, then fires the evaluation. Check results in Snowsight under AI & ML → Agents → EQ_DERIVATIVES_AGENT → Evaluations.

In [ ]:
# ============================================================================
# Upload evaluation_config.yaml to @EVALUATION_CONFIG stage
# ============================================================================
# The YAML defines which agent to evaluate, which dataset to use, and which
# metrics to compute.  We write it inline here so the lab runs end-to-end
# without a local file dependency.
from snowflake.snowpark.context import get_active_session
import os

session = get_active_session()

yaml_content = """\
# Evaluation configuration for EQ_DERIVATIVES_AGENT
# dataset: block is omitted — dataset was already created in Step 6.
# If you re-run Step 6 and get a "dataset already exists" error,
# use a new dataset_name there, then update dataset_name below to match.

evaluation:
  agent_params:
    agent_name: "EQ_DERIVATIVES_AGENT"
  run_params:
    label: "Baseline Evaluation"
    description: "Equity Derivatives HOL — baseline agent quality check"
  source_metadata:
    dataset_name: "CORTEX_AI_HOL.RAG_PIPELINE.EQ_DERIVATIVES_EVAL_DATASET"

metrics:
  answer_correctness: true
  logical_consistency: true
"""

tmp = '/tmp/evaluation_config.yaml'
with open(tmp, 'w') as f:
    f.write(yaml_content)

session.file.put(
    tmp,
    '@CORTEX_AI_HOL.RAG_PIPELINE.EVALUATION_CONFIG',
    auto_compress=False,
    overwrite=True
)
os.remove(tmp)
print("evaluation_config.yaml uploaded to @EVALUATION_CONFIG")

In [ ]:
%%sql -r step_7
-- ============================================================================
-- Run Agent Evaluation
-- ============================================================================
-- Note: can also be triggered from Snowsight → AI & ML → Agents → Evaluations

-- Start the evaluation run
CALL EXECUTE_AI_EVALUATION(
  'START',
  OBJECT_CONSTRUCT('run_name', 'eq-derivatives-baseline-v1'),
  '@CORTEX_AI_HOL.RAG_PIPELINE.EVALUATION_CONFIG/evaluation_config.yaml'
);

-- Check status (re-run this cell until STATUS = COMPLETED)
CALL EXECUTE_AI_EVALUATION(
  'STATUS',
  OBJECT_CONSTRUCT('run_name', 'eq-derivatives-baseline-v1'),
  '@CORTEX_AI_HOL.RAG_PIPELINE.EVALUATION_CONFIG/evaluation_config.yaml'
);

-- View results (run after STATUS shows COMPLETED)
-- GET_AI_EVALUATION_DATA signature: (database, schema, agent_name, run_name)
SELECT *
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
  'CORTEX_AI_HOL',
  'RAG_PIPELINE',
  'EQ_DERIVATIVES_AGENT',
  'eq-derivatives-baseline-v1'
));